# StockML exploration

This notebook only *imports* from `stockml`; all logic lives in the library. Run `python web/manage.py fetch_prices BARC.L` first (or set `allow_download=True`).

In [ ]:
from stockml.config import BacktestConfig, DataConfig, ExperimentConfig, ModelConfig
from stockml.data.cleaning import clean_prices
from stockml.data.loader import load_prices
from stockml.evaluation.backtest import run_backtest
from stockml.experiment import run_experiment
from stockml.features.pipeline import build_feature_frame, compute_indicators
from stockml.viz import charts

prices = clean_prices(load_prices("BARC.L", DataConfig(data_dir="../data"), allow_download=True))
prices.tail()

In [ ]:
charts.indicator_chart(compute_indicators(prices).loc["2020":"2021-03"], "BARC.L")

In [ ]:
X, y = build_feature_frame(prices)
charts.correlation_heatmap(X)

In [ ]:
cfg = ExperimentConfig(model=ModelConfig(models=("logistic_regression", "random_forest"), tune=False))
result = run_experiment(prices, cfg)
charts.roc_curves_chart({n: o.evaluation.roc for n, o in result.outcomes.items()})

In [ ]:
import pandas as pd

bts = {n: run_backtest(prices["Close"], o.evaluation.predictions, BacktestConfig()) for n, o in result.outcomes.items()}
equity = pd.DataFrame({n: bt.equity["strategy"] for n, bt in bts.items()})
equity[charts.BENCHMARK_LABEL] = next(iter(bts.values())).equity["buy_and_hold"]
charts.equity_curves_chart(equity, BacktestConfig().cost_bps)